# 12 — Collapse Dyadic Panel to Monadic (Recipient-Year)

Aggregates `dyadic_panel_1992_2024_oda_capped_log.csv` from sender–recipient–year dyads
to recipient–year observations, ready for baseline modelling.

**Input:** `../data/merged/dyadic_panel_1992_2024_oda_capped_log.csv`  
**Output:** `../data/merged/panel_monadic_1992_2024.csv`

| Column | Operation | Output name |
|---|---|---|
| `arms_tiv` | sum all incoming per recipient-year | `arms_tiv_total` |
| `bilateral_oda` | sum all incoming per recipient-year | `oda_total` |
| `econ_neocol_score` | sum raw values per recipient-year, then `log1p(sum × 1e9)` | `econ_neocol_score_total` |
| `colonial_tie` | max per recipient-year (1 if any sender) | `colonial_tie_flag` |
| recipient-level controls | first value (dedup only) | unchanged |

> **Note on econ_neocol_score aggregation:** Raw dyadic scores are order 1e-9 to 3.5e-5.
> They are summed first (preserving additivity across senders), then scaled by 1e9 and
> log1p-transformed — matching the dyadic `econ_neocol_score_log` convention. This
> differs from sum-of-logs and produces a non-degenerate monadic aggregate (range 0–11).

---

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load & Pre-Collapse Checks

In [ ]:
INPUT_PATH = Path("../data/merged/dyadic_panel_1992_2024_oda_capped_log.csv")
OUTPUT_PATH = Path("../data/merged/panel_monadic_1992_2024.csv")

df = pd.read_csv(INPUT_PATH)

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

Loaded: 115,640 rows x 15 columns
Columns: ['sender_iso3', 'recipient_iso3', 'year', 'arms_tiv', 'bilateral_oda', 'econ_neocol_score', 'colonial_tie', 'journalist_killings', 'gdp_per_capita', 'gdp_per_capita_log', 'population', 'population_log', 'armed_conflict', 'conflict_intensity', 'econ_neocol_score_log']


,sender_iso3,recipient_iso3,year,arms_tiv,bilateral_oda,econ_neocol_score,colonial_tie,journalist_killings,gdp_per_capita,gdp_per_capita_log,population,population_log,armed_conflict,conflict_intensity,econ_neocol_score_log
0,ABW,ISR,1996,17.80,NaN,NaN,0,0,20224.278908,9.914639,5692000.0,15.554572,1.0,1.0,NaN
1,AGO,CIV,2002,1.72,NaN,0.000000e+00,0,0,967.815864,6.875042,18654771.0,16.741612,1.0,1.0,0.000000
2,ALB,BFA,2011,1.20,NaN,7.020872e-12,0,0,725.024768,6.586206,16661908.0,16.628636,0.0,0.0,0.006996


In [3]:
# Each sender-recipient-year triple must be unique in the dyadic panel
n_dupes = df.duplicated(subset=["sender_iso3", "recipient_iso3", "year"]).sum()
print(f"Duplicate sender-recipient-year triples: {n_dupes}")
assert n_dupes == 0, "Unexpected duplicates in dyadic panel — investigate before collapsing."

n_expected_monadic = df[["recipient_iso3", "year"]].drop_duplicates().shape[0]
print(f"\nUnique senders:                      {df['sender_iso3'].nunique()}")
print(f"Unique recipients:                   {df['recipient_iso3'].nunique()}")
print(f"Year range:                          {df['year'].min()}-{df['year'].max()}")
print(f"Dyadic rows:                         {len(df):,}")
print(f"Expected monadic rows after collapse: {n_expected_monadic:,}")

Duplicate sender-recipient-year triples: 0

Unique senders:                      112
Unique recipients:                   213
Year range:                          1992-2024
Dyadic rows:                         115,640
Expected monadic rows after collapse: 6,358


## 2. Collapse to Monadic (Recipient-Year)

In [ ]:
# ── Aggregation specification ──────────────────────────────────────────────

# Dyadic sender-side variables — sum across all senders per recipient-year
sum_cols = {
    "arms_tiv":          "arms_tiv_total",
    "bilateral_oda":     "oda_total",
    "econ_neocol_score": "econ_neocol_score_raw_total",  # raw; log1p applied below
}

# Colonial tie — 1 if any sender held a colonial relationship with this recipient
max_cols = {
    "colonial_tie": "colonial_tie_flag",
}

# Recipient-level variables — constant within recipient-year; take first value
dedup_cols = [
    "journalist_killings",
    "gdp_per_capita",
    "gdp_per_capita_log",
    "population",
    "population_log",
    "armed_conflict",
    "conflict_intensity",
]

# Build groupby aggregation dict
agg_spec = {col: "sum"   for col in sum_cols}
agg_spec.update({col: "max"   for col in max_cols})
agg_spec.update({col: "first" for col in dedup_cols})

# ── Collapse ───────────────────────────────────────────────────────────────
monadic = (
    df
    .groupby(["recipient_iso3", "year"], as_index=False)
    .agg(agg_spec)
    .rename(columns={**sum_cols, **max_cols})
)

# ── Log-transform econ score AFTER summing raw values ─────────────────────
# Raw dyadic econ_neocol_score values are order 1e-9 to 3.5e-5.
# Scaling by 1e9 before log1p matches the dyadic econ_neocol_score_log
# convention (np.log1p(econ_neocol_score * 1e9)) and produces a
# non-degenerate distribution (range 0–11) for modelling.
monadic["econ_neocol_score_total"] = np.log1p(
    monadic["econ_neocol_score_raw_total"] * 1e9
)
monadic = monadic.drop(columns=["econ_neocol_score_raw_total"])

print(f"Monadic panel shape: {monadic.shape[0]:,} rows x {monadic.shape[1]} columns")
print(f"Columns: {monadic.columns.tolist()}")
monadic.head()

## 3. Post-Collapse Validation

In [5]:
# ── Row count ──────────────────────────────────────────────────────────────
assert monadic.shape[0] == n_expected_monadic, (
    f"Row mismatch: expected {n_expected_monadic:,}, got {monadic.shape[0]:,}"
)
print(f"Row count matches expected recipient-year pairs: {monadic.shape[0]:,}")

# ── No duplicate recipient-year rows ──────────────────────────────────────
n_dupes_out = monadic.duplicated(subset=["recipient_iso3", "year"]).sum()
assert n_dupes_out == 0, f"{n_dupes_out} duplicate recipient-year rows in output."
print(f"No recipient-year duplicates in output.")

print(f"\nYear range:        {monadic['year'].min()}-{monadic['year'].max()}")
print(f"Unique recipients: {monadic['recipient_iso3'].nunique()}")

Row count matches expected recipient-year pairs: 6,358
No recipient-year duplicates in output.

Year range:        1992-2024
Unique recipients: 213


In [6]:
# ── Missingness summary ────────────────────────────────────────────────────
miss = (
    monadic.isna().sum()
    .rename("n_missing")
    .to_frame()
    .assign(pct_missing=lambda x: (x["n_missing"] / len(monadic) * 100).round(2))
)
print("Missingness summary:\n")
print(miss.to_string())

Missingness summary:

                         n_missing  pct_missing
recipient_iso3                   0         0.00
year                             0         0.00
arms_tiv_total                   0         0.00
oda_total                        0         0.00
econ_neocol_score_total          0         0.00
colonial_tie_flag                0         0.00
journalist_killings              0         0.00
gdp_per_capita                 404         6.35
gdp_per_capita_log             404         6.35
population                     268         4.22
population_log                 268         4.22
armed_conflict                 235         3.70
conflict_intensity             235         3.70


In [7]:
# ── journalist_killings distribution ──────────────────────────────────────
jk = monadic["journalist_killings"]

print("journalist_killings distribution:")
print(f"  Total recipient-year obs  : {len(jk):,}")
print(f"  Zero share                : {(jk == 0).mean():.1%}")
print(f"  Mean (all obs)            : {jk.mean():.3f}")
print(f"  Mean (non-zero only)      : {jk[jk > 0].mean():.2f}")
print(f"  Max                       : {jk.max():.0f}")
print(f"  Variance / Mean ratio     : {(jk.var() / jk.mean()):.1f}x  (overdispersion check)")
print()
print(jk.describe())

journalist_killings distribution:
  Total recipient-year obs  : 6,358
  Zero share                : 88.7%
  Mean (all obs)            : 0.369
  Mean (non-zero only)      : 3.25
  Max                       : 82
  Variance / Mean ratio     : 14.7x  (overdispersion check)

count    6358.000000
mean        0.368827
std         2.332112
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        82.000000
Name: journalist_killings, dtype: float64


## 4. Save Output

In [8]:
monadic.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")
print(f"Final shape: {monadic.shape[0]:,} rows x {monadic.shape[1]} columns")

Saved to ../data/merged/panel_monadic_1992_2024.csv
Final shape: 6,358 rows x 13 columns


## 5. Rename Files in `data/merged/`

Renames existing dyadic panel files so the `dyadic_` prefix clearly distinguishes them
from the new `panel_monadic_*` output. Existing content is unchanged.

> **One-time operation.** If re-running this notebook after rename, update `INPUT_PATH`
> above to `dyadic_panel_1992_2024_oda_capped_log.csv`.

In [9]:
MERGED_DIR = Path("../data/merged")

# ── State before rename ────────────────────────────────────────────────────
print("BEFORE:")
for f in sorted(MERGED_DIR.iterdir()):
    print(f"  {f.name}")

# Maps current filename → new filename
# Goal: add dyadic_ prefix and normalise separators to underscores
rename_map = {
    "panel_with_controls_1992_2024_pre_oda_floor.csv":
        "dyadic_panel_1992_2024_pre_oda_floor.csv",
    "panel_with_controls_1992_2024-ODACAPPED.csv":
        "dyadic_panel_1992_2024_oda_capped.csv",
    "panel_with_controls_1992_2024-ODACAPPED-LOG.csv":
        "dyadic_panel_1992_2024_oda_capped_log.csv",
}

print("\nRenames:")
for old_name, new_name in rename_map.items():
    old_path = MERGED_DIR / old_name
    new_path = MERGED_DIR / new_name
    if old_path.exists():
        old_path.rename(new_path)
        print(f"  {old_name}\n    → {new_name}")
    elif new_path.exists():
        print(f"  SKIP (already renamed): {new_name}")
    else:
        print(f"  WARNING — not found: {old_name}")

# ── State after rename ─────────────────────────────────────────────────────
print("\nAFTER:")
for f in sorted(MERGED_DIR.iterdir()):
    print(f"  {f.name}")

BEFORE:
  panel_monadic_1992_2024.csv
  panel_with_controls_1992_2024-ODACAPPED-LOG.csv
  panel_with_controls_1992_2024-ODACAPPED.csv
  panel_with_controls_1992_2024_pre_oda_floor.csv

Renames:
  panel_with_controls_1992_2024_pre_oda_floor.csv
    → dyadic_panel_1992_2024_pre_oda_floor.csv
  panel_with_controls_1992_2024-ODACAPPED.csv
    → dyadic_panel_1992_2024_oda_capped.csv
  panel_with_controls_1992_2024-ODACAPPED-LOG.csv
    → dyadic_panel_1992_2024_oda_capped_log.csv

AFTER:
  dyadic_panel_1992_2024_oda_capped.csv
  dyadic_panel_1992_2024_oda_capped_log.csv
  dyadic_panel_1992_2024_pre_oda_floor.csv
  panel_monadic_1992_2024.csv
